<a href="https://colab.research.google.com/github/hawaka6667-dev/gpt-sovits-colab/blob/main/Colab_WebUI_Optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ GPT-SoVITS Colab Pro (T4 稳定优化版)

基于已实测跑通的 Colab T4 配置重构优化，包含：
- **开箱即用依赖锁**：内置 `torch==2.11.0+cu126` + `torchaudio==2.11.0+cu126` 与 `fastapi==0.115.12` + `starlette<0.47.0` 补丁，彻底解决 CUDA 版本冲突与 Gradio 500 报错。
- **Google Drive 云盘缓存**：自动缓存 HuggingFace 增量模型（v4/v5 等），支持持久化备份自训练权重、参考音频与输出配音。
- **显式计时 & 可视化进度**：实时 HTML 进度条 + 分阶段耗时统计表。
- **默认直达图 3 界面**：支持表单一键切换 **图 3（纯配音推理）** / **图 1（总控制台）** / **图 2（UVR5 音频处理）**。

## Step 1. 硬件体检 & Google Drive 云盘缓存配置

In [ ]:
#@title 🔍 1. 检查 GPU/磁盘 并配置 Google Drive 云盘缓存 { display-mode: "form" }
ENABLE_GDRIVE_CACHE = True #@param {type:"boolean"}
DRIVE_WORKSPACE = "/content/drive/MyDrive/GPT-SoVITS-Workspace" #@param {type:"string"}

import os, json, time, shutil, subprocess
from IPython.display import display, HTML

t0 = time.perf_counter()

# 1) GPU 信息检测
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
except Exception as e:
    gpu_info = f"⚠️ 未检测到 NVIDIA GPU ({e}) — 请在 [代码执行程序 -> 更改运行时类型] 中选择 T4 GPU"

# 2) 本地磁盘检测
total_c, used_c, free_c = shutil.disk_usage("/content")
local_disk_str = f"{free_c/1e9:.1f} GB 可用 / {total_c/1e9:.1f} GB 总计"

# 3) Google Drive 挂载与缓存目录初始化
drive_status = "未启用云盘缓存"
if ENABLE_GDRIVE_CACHE:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    for sub in ["outputs", "ref_audios", "weights_backup", "hf_cache"]:
        os.makedirs(os.path.join(DRIVE_WORKSPACE, sub), exist_ok=True)
    total_d, used_d, free_d = shutil.disk_usage("/content/drive/MyDrive")
    drive_status = f"✅ 已挂载 ({free_d/1e9:.1f} GB 可用 / {total_d/1e9:.1f} GB 总计) -> <code>{DRIVE_WORKSPACE}</code>"

cfg = {
    "enable_gdrive_cache": ENABLE_GDRIVE_CACHE,
    "drive_workspace": DRIVE_WORKSPACE,
    "hf_cache": os.path.join(DRIVE_WORKSPACE, "hf_cache") if ENABLE_GDRIVE_CACHE else ""
}
with open("/content/.gsv_colab_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

elapsed = time.perf_counter() - t0
display(HTML(f"""
<div style='border:1px solid #38bdf8;border-radius:8px;padding:12px 16px;background:#0f172a;color:#e2e8f0;font-family:sans-serif;'>
  <h4 style='margin:0 0 8px 0;color:#38bdf8;'>🖥️ 环境体检完成 (耗时 {elapsed:.2f}s)</h4>
  <div><b>GPU:</b> {gpu_info}</div>
  <div><b>Colab 本地空间:</b> {local_disk_str}</div>
  <div><b>云盘缓存:</b> {drive_status}</div>
</div>
"""))

## Step 2. 环境初始化 & 依赖修复（含实时进度与计时）
> **说明**：拆分为 **2A** 和 **2B** 两步。首次运行 **2A** 安装 Condacolab 时可能会自动重启一次内核（属正常现象），重启完成后再点 **2B** 即可。

In [ ]:
#@title ⚙️ 2A. 初始化 Condacolab（已安装会自动跳过） { display-mode: "form" }
import os, time, shutil
t0 = time.perf_counter()
if os.path.exists("/usr/local/conda-meta") and shutil.which("conda"):
    print(f"✅ Conda 已就绪，跳过安装 (耗时 {time.perf_counter()-t0:.2f}s)")
else:
    print("⏳ 正在安装 condacolab (预计 ~2 分钟，完成后可能自动重启内核)...")
    get_ipython().run_line_magic("pip", "install -q condacolab")
    import condacolab
    condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")
    print(f"✅ Condacolab 安装完成 (耗时 {time.perf_counter()-t0:.1f}s)")

In [ ]:
#@title 🚀 2B. 一键部署 GPT-SoVITS + 稳定依赖锁（带进度条与分阶段计时） { display-mode: "form" }
MODEL_SOURCE = "HF" #@param ["HF", "HF-Mirror", "ModelScope"]
DOWNLOAD_UVR5 = True #@param {type:"boolean"}
FORCE_REINSTALL = False #@param {type:"boolean"}

import os, json, time, shutil, subprocess
from IPython.display import display, HTML

cfg_path = "/content/.gsv_colab_config.json"
cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}
drive_ws = cfg.get("drive_workspace", "")
use_drive = cfg.get("enable_gdrive_cache", False) and os.path.exists("/content/drive/MyDrive")

TOTAL_STEPS = 5
timings = []
overall_start = time.perf_counter()
progress_handle = display(HTML("初始化中..."), display_id=True)

def fmt_time(sec):
    m, s = divmod(int(sec), 60)
    return f"{m:02d}:{s:02d} ({sec:.1f}s)"

def update_ui(step_idx, title, status_text, color="#38bdf8"):
    pct = int((step_idx / TOTAL_STEPS) * 100)
    total_el = time.perf_counter() - overall_start
    html = f"""
    <div style='border:1px solid {color};border-radius:8px;padding:12px 16px;background:#0f172a;color:#e2e8f0;font-family:sans-serif;'>
      <div style='display:flex;justify-content:space-between;margin-bottom:6px;'>
        <b>[{step_idx}/{TOTAL_STEPS}] {title}</b>
        <span>⏱️ 总耗时: {fmt_time(total_el)}</span>
      </div>
      <div style='background:#1e293b;border-radius:6px;height:14px;overflow:hidden;margin-bottom:8px;'>
        <div style='width:{pct}%;background:{color};height:100%;transition:width 0.3s;'></div>
      </div>
      <div style='font-size:13px;color:#cbd5e1;'>{status_text}</div>
    </div>
    """
    progress_handle.update(HTML(html))

def run_cmd(cmd, env=None):
    proc = subprocess.Popen(
        ["bash", "-lc", cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    tail = []
    for line in proc.stdout:
        clean = line.rstrip()
        if clean:
            print(clean)
            tail.append(clean)
            if len(tail) > 5: tail.pop(0)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"命令执行失败 (exit={rc}): {cmd}\n最近输出: " + " | ".join(tail))

env_vars = os.environ.copy()
if use_drive and cfg.get("hf_cache"):
    env_vars["HF_HOME"] = cfg["hf_cache"]

# Step 1: Clone 源码
t1 = time.perf_counter()
update_ui(1, "获取 GPT-SoVITS 源码", "正在检查 / 克隆 RVC-Boss/GPT-SoVITS...")
if FORCE_REINSTALL and os.path.exists("/content/GPT-SoVITS"):
    shutil.rmtree("/content/GPT-SoVITS")
if not os.path.exists("/content/GPT-SoVITS"):
    run_cmd("git clone https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS", env=env_vars)
else:
    print("✅ /content/GPT-SoVITS 已存在，跳过克隆")
timings.append(("1. 源码准备", time.perf_counter() - t1))

# Step 2: Conda 环境
t2 = time.perf_counter()
update_ui(2, "准备 GPTSoVITS Python 3.10 环境", "检查或创建 /usr/local/envs/GPTSoVITS...")
if not os.path.exists("/usr/local/envs/GPTSoVITS/bin/python"):
    run_cmd("conda create -n GPTSoVITS python=3.10 -y && source activate GPTSoVITS && pip install -q ipykernel", env=env_vars)
else:
    print("✅ GPTSoVITS Conda 环境已存在，跳过创建")
timings.append(("2. Conda 环境", time.perf_counter() - t2))

# Step 3: 运行 install.sh
t3 = time.perf_counter()
marker = "/content/GPT-SoVITS/.colab_installed_ok"
update_ui(3, "安装基础依赖与下载预训练模型", "正在执行 install.sh (CU126)，约需 1~2 分钟...")
if FORCE_REINSTALL or not os.path.exists(marker):
    uvr_flag = "--download-uvr5" if DOWNLOAD_UVR5 else ""
    run_cmd(f"cd /content/GPT-SoVITS && source activate GPTSoVITS && bash install.sh --device CU126 --source {MODEL_SOURCE} {uvr_flag}", env=env_vars)
    open(marker, "w").write("ok")
else:
    print("✅ 基础安装已完成，跳过重复运行 install.sh")
timings.append(("3. 基础依赖与底模", time.perf_counter() - t3))

# Step 4: 版本冲突锁定补丁 (torch 2.11.0+cu126 & fastapi 0.115.12)
t4 = time.perf_counter()
update_ui(4, "应用版本对齐补丁", "锁定 torch/torchaudio==2.11.0+cu126 与 fastapi==0.115.12 + starlette<0.47.0...")
patch_marker = "/content/GPT-SoVITS/.colab_patched_ok"
if FORCE_REINSTALL or not os.path.exists(patch_marker):
    run_cmd(
        "/usr/local/envs/GPTSoVITS/bin/pip install -q torch==2.11.0+cu126 torchaudio==2.11.0+cu126 "
        "--index-url https://download.pytorch.org/whl/cu126 --extra-index-url https://pypi.org/simple && "
        "/usr/local/envs/GPTSoVITS/bin/pip install -q 'fastapi==0.115.12' 'starlette<0.47.0'",
        env=env_vars
    )
    open(patch_marker, "w").write("ok")
else:
    print("✅ 依赖补丁已应用，跳过")
timings.append(("4. CUDA/FastAPI 对齐", time.perf_counter() - t4))

# Step 5: 自检 + 恢复云盘工作区数据
t5 = time.perf_counter()
update_ui(5, "自检环境 & 关联云盘缓存", "正在校验 PyTorch/TorchAudio/FastAPI 兼容性...")
run_cmd(
    "/usr/local/envs/GPTSoVITS/bin/python -c \"import torch, torchaudio, fastapi, starlette, gradio; "
    "print('Verify OK -> torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'torchaudio:', torchaudio.__version__, 'fastapi:', fastapi.__version__, 'starlette:', starlette.__version__)\"",
    env=env_vars
)
if use_drive:
    ref_local = "/content/GPT-SoVITS/ref_audios"
    ref_drive = os.path.join(drive_ws, "ref_audios")
    if not os.path.exists(ref_local):
        os.symlink(ref_drive, ref_local)
    wb_dir = os.path.join(drive_ws, "weights_backup")
    if os.path.exists(wb_dir):
        run_cmd(f"cp -rn {wb_dir}/* /content/GPT-SoVITS/ 2>/dev/null || true")
timings.append(("5. 自检与云盘联动", time.perf_counter() - t5))

total_sec = time.perf_counter() - overall_start
rows = "".join(f"<tr><td style='padding:4px 10px;'>{k}</td><td style='padding:4px 10px;text-align:right;'>{fmt_time(v)}</td></tr>" for k, v in timings)
progress_handle.update(HTML(f"""
<div style='border:1px solid #22c55e;border-radius:8px;padding:12px 16px;background:#0f172a;color:#e2e8f0;font-family:sans-serif;'>
  <h4 style='margin:0 0 8px 0;color:#22c55e;'>✅ 部署与兼容性修复全部完成！(总耗时: {fmt_time(total_sec)})</h4>
  <table style='width:100%;max-width:460px;border-collapse:collapse;font-size:13px;'>
    <thead><tr style='border-bottom:1px solid #334155;color:#94a3b8;'><th style='text-align:left;padding:4px 10px;'>阶段</th><th style='text-align:right;padding:4px 10px;'>耗时</th></tr></thead>
    <tbody>{rows}</tbody>
  </table>
</div>
"""))

## Step 3. 启动 WebUI（默认直达图 3 语音合成推理界面）

In [ ]:
#@title 🎙️ 启动选定界面（点击日志中的 https://xxxx.gradio.live 即可使用） { display-mode: "form" }
LAUNCH_MODE = "🚀 图3: 语音合成与克隆推理 (inference_webui.py)" #@param ["🚀 图3: 语音合成与克隆推理 (inference_webui.py)", "🎛️ 图1: 完整总控制台 (webui.py)", "🎵 图2: UVR5 人声分离/去混响 (tools/uvr5/webui.py)"]
UI_LANGUAGE = "zh_CN" #@param ["zh_CN", "Auto", "en_US"]
ENABLE_PUBLIC_SHARE = True #@param {type:"boolean"}

import os, json, time, datetime
cfg_path = "/content/.gsv_colab_config.json"
cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}

hf_env = f"export HF_HOME='{cfg['hf_cache']}' && " if cfg.get("enable_gdrive_cache") and cfg.get("hf_cache") and os.path.exists("/content/drive/MyDrive") else ""
share_val = "True" if ENABLE_PUBLIC_SHARE else "False"

if "图3" in LAUNCH_MODE:
    py_cmd = f"python GPT_SoVITS/inference_webui.py {UI_LANGUAGE}"
elif "图2" in LAUNCH_MODE:
    py_cmd = f"python tools/uvr5/webui.py \"cuda\" True 9873 {share_val}"
else:
    py_cmd = f"python webui.py {UI_LANGUAGE}"

print(f"🕒 启动时间: {datetime.datetime.now().strftime('%H:%M:%S')} | 模式: {LAUNCH_MODE}")
if cfg.get("enable_gdrive_cache"):
    print(f"💾 云盘参考音频目录已映射到: /content/GPT-SoVITS/ref_audios -> {cfg['drive_workspace']}/ref_audios")
print("⏳ 等待下方打印出 https://xxxx.gradio.live 公网链接后点击打开...\n")

t_run = time.perf_counter()
try:
    get_ipython().system(f"cd /content/GPT-SoVITS && source activate GPTSoVITS && {hf_env}export is_share={share_val} && {py_cmd}")
finally:
    dt = time.perf_counter() - t_run
    print(f"\n🛑 WebUI 已停止，本次运行时长: {int(dt//60):02d}分{int(dt%60):02d}秒")

## Step 4. 备份成果到云盘 & 在线试听生成音频
> 停掉 Step 3 后运行下方单元格，可一键把生成的语音和训练权重备份到 Google Drive，并直接在网页里试听。

In [ ]:
#@title 💾 同步输出/权重到云盘 并试听最新生成的音频 { display-mode: "form" }
MAX_PREVIEW_FILES = 5 #@param {type:"integer"}

import os, glob, json, time, shutil
from IPython.display import display, HTML, Audio

t0 = time.perf_counter()
cfg_path = "/content/.gsv_colab_config.json"
cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}
gsv_root = "/content/GPT-SoVITS"

# 1) 同步到 Google Drive
if cfg.get("enable_gdrive_cache") and os.path.exists("/content/drive/MyDrive"):
    drive_ws = cfg["drive_workspace"]
    out_local = os.path.join(gsv_root, "output")
    out_drive = os.path.join(drive_ws, "outputs")
    copied_out = 0
    if os.path.exists(out_local):
        for root, _, files in os.walk(out_local):
            for fn in files:
                src = os.path.join(root, fn)
                rel = os.path.relpath(src, out_local)
                dst = os.path.join(out_drive, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
                    shutil.copy2(src, dst)
                    copied_out += 1
    copied_w = 0
    wb_dir = os.path.join(drive_ws, "weights_backup")
    for folder in glob.glob(os.path.join(gsv_root, "*_weights*")):
        fname = os.path.basename(folder)
        for root, _, files in os.walk(folder):
            for fn in files:
                if fn.endswith((".pth", ".ckpt")):
                    src = os.path.join(root, fn)
                    rel = os.path.relpath(src, gsv_root)
                    dst = os.path.join(wb_dir, rel)
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
                        shutil.copy2(src, dst)
                        copied_w += 1
    print(f"☁️ 云盘同步完成 (耗时 {time.perf_counter()-t0:.2f}s): 新增输出音频 {copied_out} 个，模型权重 {copied_w} 个 -> {drive_ws}")
else:
    print("ℹ️ 未挂载 Google Drive，跳过云盘同步。")

# 2) 扫描并试听最新音频
audio_files = []
for ext in ("*.wav", "*.flac", "*.mp3"):
    audio_files.extend(glob.glob(os.path.join(gsv_root, "output", "**", ext), recursive=True))
audio_files.sort(key=os.path.getmtime, reverse=True)

if not audio_files:
    print("📭 output 目录暂无生成的音频文件。")
else:
    print(f"🎧 找到 {len(audio_files)} 个音频，展示最近 {min(MAX_PREVIEW_FILES, len(audio_files))} 个：")
    for p in audio_files[:MAX_PREVIEW_FILES]:
        sz = os.path.getsize(p) / 1024
        display(HTML(f"<div style='margin-top:8px;font-size:13px;'><b>{os.path.basename(p)}</b> ({sz:.1f} KB) — <code>{p}</code></div>"))
        display(Audio(p))

## 📚 内置工具探索速查（含“图 2 为何能延长音频”）

| 界面/模块 | 核心用途 | 实用玩法提示 |
| :--- | :--- | :--- |
| **图 3 (`inference_webui.py`)** | **主力配音与声音克隆** | 传 3~10 秒干声作参考音频，搭配文本即可合成；不确定参数时 `top_k / top_p / temperature` 用默认即可。 |
| **图 2 (`UVR5`)** | **人声分离 & 去混响/回声** | **为什么感觉能“延长音频”？** 当提取伴奏/混响残差轨（如 `MDX-Net` / `DeEcho` 分离出的非干声部分）时，会保留空间尾音与延迟回响，听感就像音尾被拉长；反之取干声轨能把带回音的素材洗干净再喂给图 3。 |
| **图 1 -> `0b-语音切分`** | **长音频自动切段** | 扔进一整段访谈或视频录音，自动按静音切成 4~10 秒片段，完美挑出高质量参考音频。 |
| **图 1 -> `0d-ASR 打标`** | **自动识别音频字幕** | 配合 `1A/1B` 微调训练，自动生成参考文本与训练集标注。 |
| **图 1 -> `2-Voice Changer`** | **音色转换（变声器）** | 保留原音频的语气节奏，把声音替换成目标音色。 |